In [0]:
from pyspark.sql.functions import count, col, when, current_timestamp, lit
from datetime import datetime

catalog = "flights_project"
run_timestamp = datetime.now()

dq_results = []

# Bronze checks
bronze_count = spark.table(f"{catalog}.bronze.flights_raw").count()
dq_results.append(("bronze", "row_count", float(bronze_count), "INFO"))

In [0]:
silver_df = spark.table(f"{catalog}.silver.flights_clean")
silver_count = silver_df.count()
dq_results.append(("silver", "row_count", float(silver_count), "INFO"))

# Null rate on a critical column
null_dates = silver_df.filter(col("flight_date").isNull()).count()
null_rate = round((null_dates / silver_count) * 100, 4) if silver_count > 0 else 0
status = "PASS" if null_rate < 1.0 else "FAIL"
dq_results.append(("silver", "flight_date_null_rate_pct", null_rate, status))

# Duplicate check on natural key
dup_count = (silver_df
    .groupBy("flight_date", "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST")
    .count()
    .filter(col("count") > 1)
    .count())
status = "PASS" if dup_count == 0 else "FAIL"
dq_results.append(("silver", "duplicate_key_count", float(dup_count), status))

# Row count sanity: silver shouldn't exceed bronze
status = "PASS" if silver_count <= bronze_count else "FAIL"
dq_results.append(("silver", "row_count_vs_bronze_check", float(silver_count), status))

In [0]:
gold_tables = ["delay_by_carrier", "delay_by_airport_hour", "delay_causes_breakdown", "route_risk_features"]
for t in gold_tables:
    cnt = spark.table(f"{catalog}.gold.{t}").count()
    status = "FAIL" if cnt == 0 else "PASS"
    dq_results.append((f"gold.{t}", "row_count", float(cnt), status))

# Build the log dataframe
dq_df = spark.createDataFrame(dq_results, ["layer", "metric", "value", "status"]) \
    .withColumn("run_timestamp", lit(run_timestamp))

(dq_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{catalog}.gold.data_quality_log"))

display(dq_df)

layer,metric,value,status,run_timestamp
bronze,row_count,2360969.0,INFO,2026-08-17T18:08:22.400Z
silver,row_count,2360969.0,INFO,2026-08-17T18:08:22.400Z
silver,flight_date_null_rate_pct,0.0,PASS,2026-08-17T18:08:22.400Z
silver,duplicate_key_count,0.0,PASS,2026-08-17T18:08:22.400Z
silver,row_count_vs_bronze_check,2360969.0,PASS,2026-08-17T18:08:22.400Z
gold.delay_by_carrier,row_count,14.0,PASS,2026-08-17T18:08:22.400Z
gold.delay_by_airport_hour,row_count,3625.0,PASS,2026-08-17T18:08:22.400Z
gold.delay_causes_breakdown,row_count,1.0,PASS,2026-08-17T18:08:22.400Z
gold.route_risk_features,row_count,9629.0,PASS,2026-08-17T18:08:22.400Z


In [0]:
%sql
SELECT * FROM flights_project.gold.data_quality_log
WHERE status = 'FAIL'
ORDER BY run_timestamp DESC;

layer,metric,value,status,run_timestamp
